# Dipole deconvolution + denoising (NAFNet, GoPro-width64 fine-tuning)

조교 제공 `train_final_example.ipynb` 를 뼈대로, 빠진 학습 스캐폴딩을 채우고
**공식 NAFNet-GoPro-width64(68M) pretrained 를 받아 이 과제(dipole+noise)에 fine-tune** 한다.

- 입력 `measure = N(A x)` (dipole blur + noise), 정답 `label = x`
- test 는 제공된 `test_deconv_noise` → `test_label` 로 PSNR/SSIM 측정


## 0. 환경 설정 & 데이터 준비


In [ ]:
# 이 셀이 하는 일: 필요한 라이브러리 import + 랜덤 시드 고정
# (train_final_example 은 import/상수/config 가 비어 있어 여기서 전부 채운다)
import glob
import json
import os
import random
import shutil
import time
import warnings
import zipfile
from collections.abc import Callable
from dataclasses import dataclass, field
from enum import Enum, IntEnum
from functools import lru_cache
from pathlib import Path

import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)


In [ ]:
# Google Drive 마운트 (Colab)
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
# ── 데이터 준비: dataset.zip 을 로컬(/content)에 풀어 학습을 빠르게 한다 ──
# ★ 주의: C:\Users\...\Desktop\dataset.zip 같은 "네 PC 경로"는 Colab 이 못 읽는다.
#   Desktop 의 zip 을 아래 둘 중 하나로 Colab 에 올려라:
#     (A) Google Drive 에 업로드      → /content/drive/MyDrive/dataset.zip
#     (B) Colab 좌측 파일창(폴더 아이콘)에 드래그 → /content/dataset.zip  (세션 끝나면 사라짐)
LOCAL_DATA = Path("/content/dataset")

# zip 후보 경로 (존재하는 첫 번째를 사용). 다른 곳에 뒀으면 이 리스트에 경로만 추가하면 된다.
ZIP_CANDIDATES = [
    Path("/content/dataset.zip"),                # ★ /content 에 직접 업로드 (빠름, 이번 방식)
    Path("/content/drive/MyDrive/데이터 사이언티스트(서울대)/02. 강의자료/"
         "Digital Image Processing_이종호교수님/프로젝트실습5/실습/dataset.zip"),  # Drive 백업 위치
    Path("/content/drive/MyDrive/dataset.zip"),
]


def _count_npy(p: Path) -> int:
    return sum(1 for _ in p.rglob("*.npy")) if p.exists() else 0


def _find_root(base: Path) -> Path:
    """압축 구조가 어떻든(폴더 한 겹 더 있어도) train/ 이 든 실제 폴더를 찾는다."""
    if (base / "train").is_dir():
        return base
    for p in base.rglob("train"):
        if p.is_dir():
            return p.parent
    return base


if _count_npy(LOCAL_DATA) > 0:
    DATA_ROOT_LOCAL = _find_root(LOCAL_DATA)
    print("[건너뜀] 이미 풀려 있음:", DATA_ROOT_LOCAL)
else:
    zip_path = next((z for z in ZIP_CANDIDATES if z.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(
            "dataset.zip 을 Colab 에서 찾지 못했습니다. 다음 중 하나로 올린 뒤 이 셀을 다시 실행하세요:\n"
            "  (A) Google Drive 에 업로드  → /content/drive/MyDrive/dataset.zip\n"
            "  (B) Colab 좌측 파일창에 드래그 → /content/dataset.zip\n"
            "  (C:\\Users\\...\\Desktop\\dataset.zip 같은 PC 경로는 Colab 이 못 읽습니다.)"
        )
    print("zip 사용:", zip_path, "→ 압축 해제 중...")
    LOCAL_DATA.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(LOCAL_DATA)
    DATA_ROOT_LOCAL = _find_root(LOCAL_DATA)

print("DATA_ROOT_LOCAL =", DATA_ROOT_LOCAL)
for d in ["train", "val", "test_label", "test_deconv_noise"]:
    print(f"  {d}: {_count_npy(DATA_ROOT_LOCAL / d)} npy")


In [ ]:
# 상수 + 실험 설정 + 데이터 key
VOXEL_SIZE = (1.0, 1.0)
B0_DIR = (0.0, 1.0)
NOISE_RANGES = {
    "gaussian": (0.0, 0.1),
    "rician": (0.0, 0.15),
    "uniform": (0.0, 0.2),
    "salt_and_pepper": (0.0, 0.2),
}


class DataKey(IntEnum):
    Label = 0
    Measure = 1
    Name = 2


@dataclass
class Config:
    # 데이터: 위 셀에서 zip 을 푼 DATA_ROOT_LOCAL 아래에서 train/val/test 를 모두 읽는다.
    # (zip 에 test_label / test_deconv_noise 가 들어 있어야 test 가 된다)
    train_dir: Path = DATA_ROOT_LOCAL / "train"
    val_dir: Path = DATA_ROOT_LOCAL / "val"
    test_label_dir: Path = DATA_ROOT_LOCAL / "test_label"
    test_measure_dir: Path = DATA_ROOT_LOCAL / "test_deconv_noise"  # dipole+noise (제공)
    # 결과(checkpoint/log)는 Drive 에 저장(영구 보관). Drive 를 안 쓰면 Path("/content/logs_final_nafnet") 로 바꾸세요.
    run_dir: Path = Path("/content/drive/MyDrive/logs_final_nafnet")
    # 학습 (68M 모델이라 batch 작게. OOM 나면 더 줄이세요)
    train_batch: int = 4
    valid_batch: int = 1
    num_workers: int = 2
    epochs: int = 15
    lr: float = 5e-5  # fine-tuning 이라 낮게
    device: torch.device = None  # 아래에서 설정


config = Config()
config.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", config.device)
print("train_dir:", config.train_dir)


In [ ]:
# 학습 보조: MetricController(지표 누적) + 시간/체크포인트 유틸
class MetricController:
    """키별로 값(텐서/스칼라)을 모아 평균/표준편차를 낸다."""

    def __init__(self) -> None:
        self.state: dict[str, list[float]] = {}

    def reset(self) -> None:
        self.state = {}

    def add(self, key: str, value) -> None:
        if isinstance(value, torch.Tensor):
            value = value.detach().cpu().flatten().tolist()
        else:
            value = [float(value)]
        self.state.setdefault(key, []).extend(value)

    def mean(self, key: str) -> float:
        return float(np.mean(self.state[key])) if self.state.get(key) else 0.0

    def std(self, key: str) -> float:
        return float(np.std(self.state[key], ddof=1)) if len(self.state.get(key, [])) > 1 else 0.0


def next_run_dir(run_dir: Path) -> Path:
    """logs_final_nafnet/00000_train 처럼 번호를 하나 올려 새 폴더 경로를 만든다."""
    os.makedirs(run_dir, exist_ok=True)
    ids = []
    for e in os.listdir(run_dir):
        try:
            ids.append(int(e.split("_")[0]))
        except ValueError:
            pass
    return run_dir / f"{max(ids, default=-1) + 1:05d}_train"


## 1. Forward model : dipole convolution + noise

$$y = \mathcal{N}\big(\underbrace{\mathcal{F}^{-1}\{D(k)\,\mathcal{F}\{x\}\}}_{\text{dipole convolution}}\big)$$

- `train` / `val` : clean image에 ** dipole + noise** 를 건다.
  - train 은 flip augmentation 후 매번 새 noise
  - val 은 파일 이름 기반 seed 로 noise 를 고정해 epoch 간 비교가 가능하게 한다
- `test` : 제공된 **`test_deconv_noise`**


### 1-1. Dipole kernel

$$D(k) = \frac{1}{3} - \frac{(k \cdot \hat{B_0})^2}{|k|^2}$$

`B0_dir = (0, 1)` -> magic angle (약 54.7도) 원뿔에서 `D = 0` 이라 그 성분은 원리적으로 복원 불가능하다.

In [ ]:
@lru_cache(maxsize=16)
def dipole_kernel(
    matrix_size: tuple[int, int],
    voxel_size: tuple[float, float] = VOXEL_SIZE,
    B0_dir: tuple[float, float] = B0_DIR,
) -> torch.Tensor:
    y = np.arange(-matrix_size[1] / 2, matrix_size[1] / 2, 1)
    x = np.arange(-matrix_size[0] / 2, matrix_size[0] / 2, 1)
    Y, X = np.meshgrid(y, x)

    X = X / (matrix_size[0] * voxel_size[0])
    Y = Y / (matrix_size[1] * voxel_size[1])

    D = 1 / 3 - (X * B0_dir[0] + Y * B0_dir[1]) ** 2 / (X**2 + Y**2 + 1e-8)
    D = np.fft.fftshift(D)
    return torch.tensor(D, dtype=torch.float32)


def dipole_forward(img: Tensor) -> Tensor:
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    img_k = torch.fft.fftn(img, dim=(-2, -1))
    return torch.fft.ifftn(img_k * kernel, dim=(-2, -1)).real


dipole_adjoint = dipole_forward  # A^T = A (D 가 real, even)

_D = dipole_kernel((256, 256))
print(f"D range [{_D.min():.4f}, {_D.max():.4f}] | |D| < 0.01 인 비율 {float((_D.abs() < 0.01).float().mean()) * 100:.2f}%")

### 1-2. Noise 4종 (Gaussian, Rician, Uniform, Salt and Pepper)

    "gaussian": (0.0, 0.1),
    "rician": (0.0, 0.15),
    "uniform": (0.0, 0.2),
    "salt_and_pepper": (0.0, 0.2)

In [ ]:
class NoisyType(str, Enum):
    Gaussian = "gaussian"
    Rician = "rician"
    Uniform = "uniform"
    SaltAndPepper = "salt_and_pepper"

    @classmethod
    def from_string(cls, value: str) -> "NoisyType":
        try:
            return cls(value)
        except ValueError as err:
            raise ValueError(f"Invalid NoisyType value: {value}. Must be one of {list(cls)} : {err}") from err

Gen = torch.Generator | None


def gaussian_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    noise = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    return img + noise


def rician_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    noise_real = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    noise_imag = torch.empty_like(img).normal_(0.0, 1.0, generator=generator) * sigma
    return torch.abs(img + noise_real + 1j * noise_imag)


def uniform_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    return img + (torch.empty_like(img).uniform_(0.0, 1.0, generator=generator) * 2.0 - 1.0) * sigma


def salt_and_pepper_noise(img: Tensor, sigma: float, generator: Gen = None) -> Tensor:
    salt_prob = sigma / 2
    pepper_prob = sigma / 2
    noisy_img = img.clone()
    total_pixels = img.numel()

    num_salt = int(total_pixels * salt_prob)
    coords = [torch.randint(0, dim, (num_salt,), generator=generator) for dim in img.shape]
    noisy_img[tuple(coords)] = img.max()

    num_pepper = int(total_pixels * pepper_prob)
    coords = [torch.randint(0, dim, (num_pepper,), generator=generator) for dim in img.shape]
    noisy_img[tuple(coords)] = 0

    return noisy_img


NOISE_FUNC: dict[NoisyType, Callable[..., Tensor]] = {
    NoisyType.Gaussian: gaussian_noise,
    NoisyType.Rician: rician_noise,
    NoisyType.Uniform: uniform_noise,
    NoisyType.SaltAndPepper: salt_and_pepper_noise,
}


class NoiseSimulator:
    """noise 종류 하나 + sigma 하나를 고정해서 적용한다."""

    def __init__(self, noisy_type: NoisyType, sigma: float) -> None:
        self.noisy_type = noisy_type
        self.sigma = sigma

    def __call__(self, img: Tensor, generator: Gen = None) -> Tensor:
        return NOISE_FUNC[self.noisy_type](img, self.sigma, generator)


class RandomNoiseSimulator:
    """이미지마다 4종 중 하나를 랜덤으로 골라 NOISE_RANGES 범위에서 sigma 를 뽑는다."""

    def __init__(self, noise_ranges: dict[str, tuple[float, float]] | None = None) -> None:
        self.noise_ranges = dict(noise_ranges) if noise_ranges is not None else dict(NOISE_RANGES)
        self.names = list(self.noise_ranges.keys())

    def _sample(self, rng) -> tuple[str, float]:
        name = rng.choice(self.names)
        low, high = self.noise_ranges[name]
        return name, rng.uniform(low, high)

    def __call__(self, img: Tensor, seed: int | None = None) -> Tensor:
        if seed is None:
            # 학습용: 매번 새로 뽑는다 (같은 이미지도 epoch 마다 다른 노이즈)
            name, sigma = self._sample(random)
            generator = None
        else:
            # 검증용: 이미지마다 항상 같은 노이즈가 나오도록 seed 를 고정한다.
            name, sigma = self._sample(random.Random(seed))
            generator = torch.Generator().manual_seed(int(seed) % (2**63 - 1))
        return NoiseSimulator(NoisyType.from_string(name), sigma)(img, generator)

    def describe(self, seed: int) -> tuple[str, float]:
        return self._sample(random.Random(seed))


class DegradationSimulator:
    """clean image -> (blur, measure, noisy_label).

    - blur        : noise 없는 dipole 결과 A x
    - measure     : 실제 측정 N(A x)          <- dipole + noise
    - noisy_label : N(x)                      <- dipole 없이 noise 만 (DnCNN augmentation 용)
    """

    def __init__(self, noise_ranges: dict[str, tuple[float, float]] | None = None) -> None:
        self.noise = RandomNoiseSimulator(noise_ranges)

    def __call__(self, label: Tensor, seed: int | None = None) -> tuple[Tensor, Tensor, Tensor]:
        blur = dipole_forward(label)
        measure = self.noise(blur, seed=seed)
        # clean branch 는 dipole branch 와 독립적인 noise 를 뽑는다 (seed 는 겹치지 않게 어긋냄)
        noisy_label = self.noise(label, seed=None if seed is None else seed ^ 0x5BF03635)
        return blur, measure, noisy_label

In [ ]:
# DataWrapper: .npy 를 (label, measure) 쌍으로 공급
#  - train/val : label 에 DegradationSimulator(dipole+noise) 를 걸어 measure 를 즉석 생성
#                (val 은 파일이름 seed 로 noise 고정 → epoch 간 비교 가능)
#  - test      : 제공된 measure(test_deconv_noise) 를 그대로 읽는다
class DataWrapper(Dataset):
    def __init__(self, label_dir, training_mode: bool, measure_dir=None, max_images=None):
        self.files = sorted(glob.glob(str(Path(label_dir) / "*.npy")))
        if max_images is not None:
            self.files = self.files[:max_images]
        if len(self.files) == 0:
            raise FileNotFoundError(f"라벨 .npy 가 없음: {label_dir}")
        self.training_mode = training_mode
        self.measure_dir = Path(measure_dir) if measure_dir is not None else None
        self.degrade = DegradationSimulator()

    def __len__(self):
        return len(self.files)

    @staticmethod
    def _load(p) -> Tensor:
        img = torch.from_numpy(np.load(str(p))).float()
        if img.dim() == 2:
            img = img[None]  # (H,W) -> (1,H,W)
        return img

    def __getitem__(self, i):
        name = Path(self.files[i]).name
        label = self._load(self.files[i])
        if self.measure_dir is None:
            # train/val: 즉석 degradation
            if self.training_mode:
                if random.random() > 0.5:
                    label = torch.flip(label, dims=[1])
                if random.random() > 0.5:
                    label = torch.flip(label, dims=[2])
                seed = None
            else:
                seed = abs(hash(name)) % (2**31)  # val: 이름 기반 고정 seed
            _blur, measure, _noisy = self.degrade(label, seed=seed)
        else:
            # test: 제공된 measure 를 같은 이름으로 읽는다
            mp = self.measure_dir / name
            if not mp.exists():
                raise FileNotFoundError(f"measure 파일 없음: {mp}")
            measure = self._load(mp)
        return label, measure, name


def make_loader(label_dir, training_mode, measure_dir=None, batch=1, shuffle=False, max_images=None):
    ds = DataWrapper(label_dir, training_mode, measure_dir, max_images)
    ld = DataLoader(ds, batch_size=batch, shuffle=shuffle, num_workers=config.num_workers,
                    pin_memory=True, persistent_workers=config.num_workers > 0)
    return ld, ds


# 로더 준비 (conventional 셀과 NAFNet 셀이 함께 사용)
train_loader, train_ds = make_loader(config.train_dir, True, batch=config.train_batch, shuffle=True)
valid_loader, valid_ds = make_loader(config.val_dir, False, batch=config.valid_batch, shuffle=False)
print(f"train {len(train_ds)} | valid {len(valid_ds)}")


## 2. Metric (PSNR / SSIM)

In [ ]:
IMG_DIM: int = 4


class SSIMcal(torch.nn.Module):
    def __init__(self, win_size: int = 11, k1: float = 0.01, k2: float = 0.03):
        super().__init__()
        self.win_size = win_size
        self.k1, self.k2 = k1, k2
        self.register_buffer("w", torch.ones(1, 1, win_size, win_size) / win_size**2)
        np_ = win_size**2
        self.cov_norm = np_ / (np_ - 1)

    def forward(self, img: Tensor, ref: Tensor, data_range: Tensor) -> Tensor:
        data_range = data_range[:, None, None, None]
        C1 = (self.k1 * data_range) ** 2
        C2 = (self.k2 * data_range) ** 2

        w = self.w.to(img.device)
        ux = functional.conv2d(img, w)
        uy = functional.conv2d(ref, w)
        uxx = functional.conv2d(img * img, w)
        uyy = functional.conv2d(ref * ref, w)
        uxy = functional.conv2d(img * ref, w)

        vx = self.cov_norm * (uxx - ux * ux)
        vy = self.cov_norm * (uyy - uy * uy)
        vxy = self.cov_norm * (uxy - ux * uy)

        A1 = 2 * ux * uy + C1
        A2 = 2 * vxy + C2
        B1 = ux**2 + uy**2 + C1
        B2 = vx + vy + C2

        return torch.mean((A1 * A2) / (B1 * B2), dim=[2, 3], keepdim=True)


ssim_cal = SSIMcal()


def calculate_ssim(img: Tensor, ref: Tensor, mask: Tensor | None = None) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")

    if mask is None:
        img_mask, ref_mask = img, ref
    else:
        if mask.dim() != IMG_DIM:
            raise ValueError("Mask must be 4D.")
        img_mask, ref_mask = img * mask, ref * mask

    ones = torch.ones(ref.shape[0], device=ref.device)
    return ssim_cal.forward(img_mask, ref_mask, ones)


def calculate_psnr(img: Tensor, ref: Tensor, mask: Tensor | None = None) -> Tensor:
    if not (img.dim() == IMG_DIM and ref.dim() == IMG_DIM):
        raise ValueError("All tensors must be 4D.")

    if mask is not None:
        if mask.dim() != IMG_DIM:
            raise ValueError("Mask must be 4D.")
        img_mask, ref_mask = img * mask, ref * mask
        mse = torch.sum((img_mask - ref_mask) ** 2, dim=(1, 2, 3)) / torch.sum(mask, dim=(1, 2, 3))
    else:
        mse = torch.mean(functional.mse_loss(img, ref, reduction="none"), dim=(1, 2, 3), keepdim=True)

    img_max = torch.amax(ref, dim=(1, 2, 3), keepdim=True)
    return 10 * torch.log10(img_max**2 / (mse + 1e-12))


def finalize(pred: Tensor) -> Tensor:
    """모든 방법에 동일하게 적용하는 후처리. label 이 [0,1] 이므로 clip 한다."""
    return pred.clamp(0.0, 1.0) if config.clip_output else pred


def psnr_ssim_np(img: np.ndarray, ref: np.ndarray) -> tuple[float, float]:
    def _t(arr: np.ndarray) -> Tensor:
        return torch.from_numpy(np.ascontiguousarray(arr)).float()[None, None]

    _img, _ref = _t(img), _t(ref)
    return float(calculate_psnr(_img, _ref).item()), float(calculate_ssim(_img, _ref).mean().item())

### 2-1. Train synthetic pair 확인

`train` split 은 매 epoch 새로 noise 를 뽑으므로, 같은 영상이라도 매번 다른 degradation 이 걸린다.

## 3. Conventional methods : mean / median / adaptive filter + Wiener filter

학습 없이, forward model (`dipole_kernel`) 을 안다는 가정만으로 복원한다.

1. **denoise** : mean / median / adaptive filter
2. **deconvolve** : Wiener filter
   $$\hat{x} = \mathcal{F}^{-1}\Big\{\frac{D}{D^2 + K}\,\mathcal{F}\{y\}\Big\}$$
   `K` 는 validation set 에서 PSNR 이 최대가 되도록 sweep 해서 고른다.

In [ ]:
def _as_bchw(img: Tensor) -> tuple[Tensor, int]:
    dim = img.dim()
    if dim == 2:
        return img[None, None], dim
    if dim == 3:
        return img[None], dim
    if dim == 4:
        return img, dim
    raise ValueError(f"unsupported image dim: {dim}")


def _restore_dim(img: Tensor, dim: int) -> Tensor:
    if dim == 2:
        return img[0, 0]
    if dim == 3:
        return img[0]
    return img


def mean_filter(img: Tensor, kernel_size: int = 3) -> Tensor:
    x, dim = _as_bchw(img)
    pad = kernel_size // 2
    x = functional.pad(x, (pad, pad, pad, pad), mode="reflect")
    return _restore_dim(functional.avg_pool2d(x, kernel_size=kernel_size, stride=1), dim)


def median_filter(img: Tensor, kernel_size: int = 3) -> Tensor:
    x, dim = _as_bchw(img)
    pad = kernel_size // 2
    x = functional.pad(x, (pad, pad, pad, pad), mode="reflect")
    patches = x.unfold(2, kernel_size, 1).unfold(3, kernel_size, 1)
    out = patches.reshape(*patches.shape[:4], -1).median(dim=-1).values
    return _restore_dim(out, dim)


def adaptive_filter(img: Tensor, kernel_size: int = 5, noise_var: Tensor | float | None = None) -> Tensor:
    x, dim = _as_bchw(img)
    pad = kernel_size // 2
    xp = functional.pad(x, (pad, pad, pad, pad), mode="reflect")

    local_mean = functional.avg_pool2d(xp, kernel_size=kernel_size, stride=1)
    local_sq = functional.avg_pool2d(xp.pow(2), kernel_size=kernel_size, stride=1)
    local_var = (local_sq - local_mean.pow(2)).clamp_min(0.0)

    if noise_var is None:
        noise_var = local_var.flatten(2).median(dim=-1).values[:, :, None, None]

    ratio = (noise_var / local_var.clamp_min(1e-8)).clamp(max=1.0)
    return _restore_dim(x - ratio * (x - local_mean), dim)


def wiener_deconv(img: Tensor, K: float) -> Tensor:
    """(1/D) * D^2/(D^2 + K) = D / (D^2 + K). D 가 실수라 0 나눗셈 없이 바로 쓸 수 있다."""
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    w = kernel / (kernel**2 + K)
    img_k = torch.fft.fftn(img, dim=(-2, -1))
    return torch.fft.ifftn(img_k * w, dim=(-2, -1)).real


def tkd(img: Tensor, clip: float = 5.0) -> Tensor:
    kernel = dipole_kernel(tuple(img.shape[-2:])).to(img.device)
    kernel_inv = torch.clip(1 / kernel, min=-clip, max=clip)
    img_k = torch.fft.fftn(img, dim=(-2, -1))
    return torch.fft.ifftn(img_k * kernel_inv, dim=(-2, -1)).real


BASELINE_KERNEL: int = 3
ADAPTIVE_KERNEL: int = 5

PREFILTERS: dict[str, Callable[[Tensor], Tensor]] = {
    "none": lambda x: x,
    "mean": lambda x: mean_filter(x, kernel_size=BASELINE_KERNEL),
    "median": lambda x: median_filter(x, kernel_size=BASELINE_KERNEL),
    "adaptive": lambda x: adaptive_filter(x, kernel_size=ADAPTIVE_KERNEL),
}
PREFILTER_LABEL: dict[str, str] = {
    "none": "Wiener only",
    "mean": f"Mean {BASELINE_KERNEL}x{BASELINE_KERNEL} + Wiener",
    "median": f"Median {BASELINE_KERNEL}x{BASELINE_KERNEL} + Wiener",
    "adaptive": f"Adaptive {ADAPTIVE_KERNEL}x{ADAPTIVE_KERNEL} + Wiener",
}

### 3-1. Validation set 으로 Wiener `K` 튜닝

### 3-2. Test set 에서 conventional methods results

## 4. Methods using network

공통 building block 은 `train_deconvolution_example.ipynb` 의 U-Net,
`train_denoising_example.ipynb` 의 DnCNN 을 그대로 쓴다.

| | 방법 | 설명 |
|---|---|---|
| **A** | End2End U-Net | measure -> label 을 그냥 U-Net 으로 회귀 |
| **B** | DnCNN denoise + Wiener deconvolution | measurement domain 에서 noise 만 학습으로 제거하고, deconvolution 은 Wiener filter |

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_chans: int, out_chans: int) -> None:
        super().__init__()
        self.layers = nn.Sequential(
            nn.Conv2d(in_chans, out_chans, kernel_size=3, padding=1),
            nn.GroupNorm(4, out_chans),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_chans, out_chans, kernel_size=3, padding=1),
            nn.GroupNorm(4, out_chans),
            nn.SiLU(inplace=True),
        )

    def forward(self, x: Tensor) -> Tensor:
        return self.layers(x)


def create_down_sample_layers(in_chans: int, chans: int, num_pool_layers: int) -> nn.ModuleList:
    layers = nn.ModuleList([ConvBlock(in_chans, chans)])
    ch = chans
    for _ in range(num_pool_layers - 1):
        layers.append(ConvBlock(ch, ch * 2))
        ch *= 2
    return layers


def create_up_sample_layers(chans: int, num_pool_layers: int) -> nn.ModuleList:
    layers = nn.ModuleList()
    ch = chans * (2 ** (num_pool_layers - 1))
    for _ in range(num_pool_layers - 1):
        layers.append(ConvBlock(ch * 2, ch // 2))
        ch //= 2
    layers.append(ConvBlock(ch * 2, ch))
    return layers


class Unet(nn.Module):
    def __init__(self, in_chans: int = 1, out_chans: int = 1, chans: int = 32, num_pool_layers: int = 4) -> None:
        super().__init__()
        self.in_chans = in_chans
        self.out_chans = out_chans
        self.num_pool_layers = num_pool_layers

        self.down_sample_layers = create_down_sample_layers(in_chans, chans, num_pool_layers)
        self.bottleneck_conv = ConvBlock(chans * (2 ** (num_pool_layers - 1)), chans * (2 ** (num_pool_layers - 1)))
        self.up_sample_layers = create_up_sample_layers(chans, num_pool_layers)
        self.final_conv = nn.Sequential(
            nn.Conv2d(chans, chans, kernel_size=3, padding=1),
            nn.GroupNorm(4, chans),
            nn.SiLU(inplace=True),
            nn.Conv2d(chans, out_chans, kernel_size=1, padding=0),
        )

    def forward(self, x: Tensor) -> Tensor:
        stack = []
        output = x
        for layer in self.down_sample_layers:
            output = layer(output)
            stack.append(output)
            output = functional.max_pool2d(output, kernel_size=2)

        output = self.bottleneck_conv(output)

        for layer in self.up_sample_layers:
            downsampled_output = stack.pop()
            output = functional.interpolate(
                output, size=downsampled_output.shape[-2:], mode="bilinear", align_corners=False
            )
            output = torch.cat([output, downsampled_output], dim=1)
            output = layer(output)

        return self.final_conv(output)


class DnCNN(nn.Module):
    def __init__(self, channels: int, num_of_layers: int, kernel_size: int, padding: int, features: int) -> None:
        super().__init__()
        layers: list[nn.Module] = [
            nn.Conv2d(channels, features, kernel_size=kernel_size, padding=padding, bias=False),
            nn.SiLU(inplace=True),
        ]
        for _ in range(num_of_layers - 1):
            layers += [
                nn.Conv2d(features, features, kernel_size=kernel_size, padding=padding, bias=False),
                nn.GroupNorm(4, features),
                nn.SiLU(inplace=True),
            ]
        layers.append(nn.Conv2d(features, channels, kernel_size=kernel_size, padding=padding, bias=False))
        self.dncnn = nn.Sequential(*layers)

    def forward(self, x: Tensor) -> Tensor:
        if x.dim() != 4:
            raise ValueError(f"Input tensor must be 4D, but got {x.dim()}D tensor.")
        return x + self.dncnn(x)

## 5. NAFNet (GoPro-width64 pretrained) fine-tuning

공식 NAFNet-GoPro-width64(68M, RGB 모션블러) 를 받아, 흑백 1채널로 감싸(3채널 복제/평균)
우리 과제(dipole+noise)에 낮은 LR 로 이어 학습한다.


In [ ]:
# NAFNet (공식 GoPro 가중치와 키 호환되는 구현) + 흑백 1채널 래퍼
class LayerNorm2d(nn.Module):
    """이미지(B,C,H,W)에 채널 방향 LayerNorm."""
    def __init__(self, c, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(c))
        self.bias = nn.Parameter(torch.zeros(c))
        self.eps = eps
    def forward(self, x):
        mu = x.mean(1, keepdim=True)
        var = (x - mu).pow(2).mean(1, keepdim=True)
        x = (x - mu) / torch.sqrt(var + self.eps)
        return x * self.weight[None, :, None, None] + self.bias[None, :, None, None]


class SimpleGate(nn.Module):
    """채널 반으로 나눠 곱 (ReLU 대체)."""
    def forward(self, x):
        x1, x2 = x.chunk(2, dim=1)
        return x1 * x2


class NAFBlock(nn.Module):
    def __init__(self, c, dw_expand=2, ffn_expand=2):
        super().__init__()
        dw_c, ffn_c = c * dw_expand, c * ffn_expand
        self.norm1 = LayerNorm2d(c)
        self.conv1 = nn.Conv2d(c, dw_c, 1)
        self.conv2 = nn.Conv2d(dw_c, dw_c, 3, padding=1, groups=dw_c)
        self.sg = SimpleGate()
        self.sca = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(dw_c // 2, dw_c // 2, 1))
        self.conv3 = nn.Conv2d(dw_c // 2, c, 1)
        self.norm2 = LayerNorm2d(c)
        self.conv4 = nn.Conv2d(c, ffn_c, 1)
        self.conv5 = nn.Conv2d(ffn_c // 2, c, 1)
        self.beta = nn.Parameter(torch.zeros(1, c, 1, 1))
        self.gamma = nn.Parameter(torch.zeros(1, c, 1, 1))
    def forward(self, inp):
        x = self.norm1(inp); x = self.conv1(x); x = self.conv2(x); x = self.sg(x)
        x = x * self.sca(x); x = self.conv3(x); y = inp + x * self.beta
        x = self.norm2(y); x = self.conv4(x); x = self.sg(x); x = self.conv5(x)
        return y + x * self.gamma


class NAFNet(nn.Module):
    def __init__(self, img_channel=3, width=64, enc_blk_nums=None, middle_blk_num=1, dec_blk_nums=None):
        super().__init__()
        enc_blk_nums = enc_blk_nums or [1, 1, 1, 28]
        dec_blk_nums = dec_blk_nums or [1, 1, 1, 1]
        self.intro = nn.Conv2d(img_channel, width, 3, padding=1)
        self.ending = nn.Conv2d(width, img_channel, 3, padding=1)
        self.encoders, self.decoders = nn.ModuleList(), nn.ModuleList()
        self.downs, self.ups = nn.ModuleList(), nn.ModuleList()
        chan = width
        for num in enc_blk_nums:
            self.encoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
            self.downs.append(nn.Conv2d(chan, chan * 2, 2, 2))
            chan *= 2
        self.middle_blks = nn.Sequential(*[NAFBlock(chan) for _ in range(middle_blk_num)])
        for num in dec_blk_nums:
            self.ups.append(nn.Sequential(nn.Conv2d(chan, chan * 2, 1, bias=False), nn.PixelShuffle(2)))
            chan //= 2
            self.decoders.append(nn.Sequential(*[NAFBlock(chan) for _ in range(num)]))
        self.padder_size = 2 ** len(self.encoders)
    def forward(self, inp):
        _, _, H, W = inp.shape
        inp = self._pad(inp)
        x = self.intro(inp)
        encs = []
        for enc, down in zip(self.encoders, self.downs):
            x = enc(x); encs.append(x); x = down(x)
        x = self.middle_blks(x)
        for dec, up, skip in zip(self.decoders, self.ups, encs[::-1]):
            x = up(x); x = x + skip; x = dec(x)
        x = self.ending(x); x = x + inp
        return x[:, :, :H, :W]
    def _pad(self, x):
        _, _, h, w = x.shape
        ph = (self.padder_size - h % self.padder_size) % self.padder_size
        pw = (self.padder_size - w % self.padder_size) % self.padder_size
        return F.pad(x, (0, pw, 0, ph))


class GrayNAFNet(nn.Module):
    """흑백(1ch) 입출력을 공식 3ch NAFNet 으로 감싼다: 입력 1->3 복제, 출력 3->1 평균."""
    def __init__(self, **kw):
        super().__init__()
        self.net = NAFNet(img_channel=3, **kw)
    def forward(self, x):
        y = self.net(x.repeat(1, 3, 1, 1))
        return y.mean(dim=1, keepdim=True)


# sanity
_m = GrayNAFNet(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1, dec_blk_nums=[1, 1, 1, 1])
print(f"GrayNAFNet params: {sum(p.numel() for p in _m.net.parameters())/1e6:.3f} M (공식 67.9M 와 일치해야 함)")
del _m


In [ ]:
# 모델 준비:
#  - RESUME_FROM 이 None 이면  → 공식 GoPro-width64 pretrained 로 시작 (처음 학습)
#  - RESUME_FROM 에 경로를 주면 → 그 checkpoint 에서 이어서 학습 (resume)
#
# ★ 이어서 학습하려면: 이전 run 끝에 찍힌 RUN_DIR 을 보고 아래처럼 지정하세요.
#   RESUME_FROM = "/content/drive/MyDrive/logs_final_nafnet/00000_train/checkpoints/checkpoint_best.ckpt"
#   (그리고 config.epochs 를 원하는 만큼 더 돌리도록 두면 됩니다)
RESUME_FROM = None

model = GrayNAFNet(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1, dec_blk_nums=[1, 1, 1, 1])

if RESUME_FROM is not None:
    ck = torch.load(str(RESUME_FROM), map_location="cpu", weights_only=True)
    model.load_state_dict(ck["model_state_dict"])   # 이전 학습 가중치 전체를 그대로 이어받음
    print("이어서 학습(resume):", RESUME_FROM)
else:
    try:
        import gdown
    except ImportError:
        import subprocess, sys
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"])
        import gdown
    GOPRO_ID = "1S0PVRbyTakYY9a82kujgZLbMihfNBLfC"  # NAFNet-GoPro-width64.pth
    PRETRAINED = "/content/NAFNet-GoPro-width64.pth"
    if not os.path.exists(PRETRAINED):
        gdown.download(id=GOPRO_ID, output=PRETRAINED, quiet=False)
    raw = torch.load(PRETRAINED, map_location="cpu", weights_only=True)
    state = raw.get("params", raw.get("model_state_dict", raw))  # BasicSR 는 'params' 키에 저장
    res = model.net.load_state_dict(state, strict=False)
    print(f"GoPro pretrained 로드 | missing {len(res.missing_keys)} | unexpected {len(res.unexpected_keys)}")

model = model.to(config.device)


In [ ]:
# fine-tuning 루프
#  - USE_AMP: 초기 nan 방지를 위해 기본 off (fp32). 안정적으로 학습되면 True 로 켜서 속도/메모리 이득.
#  - grad clipping + nan-batch 건너뛰기로 발산/오염 방지.
USE_AMP = False  # nan 나면 반드시 False 로 두세요


def evaluate(model, loader, desc="valid"):
    model.eval()
    mc = MetricController()
    with torch.no_grad():
        for label, measure, _ in tqdm(loader, desc=desc, leave=False):
            label = label.to(config.device)
            measure = measure.to(config.device)
            out = model(measure)
            mc.add("psnr", calculate_psnr(out, label))
            mc.add("ssim", calculate_ssim(out, label))
    return mc.mean("psnr"), mc.mean("ssim")


def finetune(model, train_loader, valid_loader, epochs, lr):
    run_dir = next_run_dir(config.run_dir)
    os.makedirs(run_dir / "checkpoints", exist_ok=True)
    print("run dir:", run_dir)

    # 데이터 sanity: 첫 batch 가 유한한지 확인 (nan 원인이 데이터인지 빠르게 배제)
    _l, _m, _ = next(iter(train_loader))
    print(f"[sanity] measure finite={torch.isfinite(_m).all().item()} range[{_m.min():.3f},{_m.max():.3f}] | "
          f"label finite={torch.isfinite(_l).all().item()} range[{_l.min():.3f},{_l.max():.3f}]")

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and config.device.type == "cuda")
    lossf = nn.L1Loss()

    best = -1.0
    for ep in range(epochs):
        model.train()
        mc = MetricController()
        skipped = 0
        for label, measure, _ in tqdm(train_loader, desc=f"train ep{ep + 1}/{epochs}", leave=False):
            label = label.to(config.device)
            measure = measure.to(config.device)
            opt.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP and config.device.type == "cuda"):
                out = model(measure)
                loss = lossf(out, label)
            if not torch.isfinite(loss):      # nan/inf 이면 이 batch 는 건너뛴다(가중치 오염 방지)
                skipped += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 그래디언트 발산 방지
            scaler.step(opt)
            scaler.update()
            mc.add("loss", loss.detach())

        val_psnr, val_ssim = evaluate(model, valid_loader)
        msg = f"[ep {ep + 1}] loss {mc.mean('loss'):.4f} | val PSNR {val_psnr:.3f} SSIM {val_ssim:.4f}"
        if skipped:
            msg += f" | (nan/inf batch {skipped}개 건너뜀)"
        print(msg)

        if val_psnr > best:
            best = val_psnr
            torch.save(
                {"model_state_dict": model.state_dict(),
                 "model_config": {"width": 64, "enc_blk_nums": [1, 1, 1, 28],
                                  "middle_blk_num": 1, "dec_blk_nums": [1, 1, 1, 1]}},
                run_dir / "checkpoints/checkpoint_best.ckpt",
            )
            print(f"    best 갱신 (val PSNR {best:.3f}) -> 저장")
    return run_dir, best


RUN_DIR, BEST = finetune(model, train_loader, valid_loader, config.epochs, config.lr)
print("fine-tune 완료 | best val PSNR:", round(BEST, 3), "| run:", RUN_DIR)


In [ ]:
# 제공된 test_deconv_noise 로 최종 평가 + 노이즈 종류별 분석
# best checkpoint 로드
best_ckpt = RUN_DIR / "checkpoints/checkpoint_best.ckpt"
ck = torch.load(best_ckpt, map_location="cpu", weights_only=True)
test_model = GrayNAFNet(width=64, enc_blk_nums=[1, 1, 1, 28], middle_blk_num=1, dec_blk_nums=[1, 1, 1, 1])
test_model.load_state_dict(ck["model_state_dict"])
test_model = test_model.to(config.device).eval()

test_loader, test_ds = make_loader(
    config.test_label_dir, training_mode=False, measure_dir=config.test_measure_dir, batch=1, shuffle=False
)

# 노이즈 종류 메타 (분석용)
meta_path = Path(config.test_measure_dir) / "noise_meta.json"
noise_meta = json.load(open(meta_path)) if meta_path.exists() else {}

rows = []
with torch.no_grad():
    for label, measure, name in tqdm(test_loader, desc="test"):
        label = label.to(config.device)
        measure = measure.to(config.device)
        out = test_model(measure)
        nm = name[0]
        ntype = noise_meta.get(nm, {}).get("noise_type", "unknown") if isinstance(noise_meta, dict) else "unknown"
        rows.append({
            "file": nm, "noise_type": ntype,
            "psnr": calculate_psnr(out, label).item(),
            "ssim": calculate_ssim(out, label).item(),
            "psnr_in": calculate_psnr(measure, label).item(),
        })

print(f"\n=== NAFNet(GoPro fine-tuned) test 결과 (n={len(rows)}) ===")
print(f"PSNR in {np.mean([r['psnr_in'] for r in rows]):.3f} -> out {np.mean([r['psnr'] for r in rows]):.3f} dB")
print(f"SSIM out {np.mean([r['ssim'] for r in rows]):.4f}")

print("\n[노이즈 종류별 PSNR/SSIM]")
types = sorted({r["noise_type"] for r in rows})
for t in types:
    sub = [r for r in rows if r["noise_type"] == t]
    print(f"  {t:<16} n={len(sub):<3} PSNR {np.mean([r['psnr'] for r in sub]):.3f}  SSIM {np.mean([r['ssim'] for r in sub]):.4f}")

json.dump(rows, open(RUN_DIR / "test_metrics.json", "w"), indent=2)
print("\n저장:", RUN_DIR / "test_metrics.json")


In [ ]:
# 결과 시각화: measure / NAFNet 복원 / 정답 / error map (몇 장)
N_SHOW = 3
test_model.eval()
shown = 0
fig, axes = plt.subplots(N_SHOW, 4, figsize=(15, 4 * N_SHOW))
axes = np.asarray(axes).reshape(N_SHOW, 4)
with torch.no_grad():
    for label, measure, name in test_loader:
        if shown >= N_SHOW:
            break
        lab = label.to(config.device)
        mea = measure.to(config.device)
        out = test_model(mea)
        p = calculate_psnr(out, lab).item()
        L = lab.cpu().numpy().squeeze(); M = mea.cpu().numpy().squeeze()
        O = out.cpu().numpy().squeeze(); E = np.abs(O - L)
        vmax = float(np.percentile(L, 98) * 1.2)
        panels = [(M, "measure (input)", "gray", np.percentile(M, 1), np.percentile(M, 99)),
                  (O, f"NAFNet out\nPSNR {p:.2f}dB", "gray", 0, vmax),
                  (L, "label (GT)", "gray", 0, vmax),
                  (E, "|error|", "magma", 0, max(E.max(), 1e-6))]
        for ax, (im, t, cm, lo, hi) in zip(axes[shown], panels):
            ax.imshow(im, cmap=cm, vmin=lo, vmax=hi); ax.set_title(t, fontsize=9); ax.axis("off")
        shown += 1
fig.suptitle("NAFNet (GoPro-width64 fine-tuned) on test_deconv_noise")
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()
